# Telugu ASR — File 3 of 3: Full-Scale CTC Fine-Tuning
## EXPERIMENT 3: Stronger SpecAugment (Fallback)
## `facebook/wav2vec2-xls-r-300m` → Telugu · MSCIL Dataset

---

### Context
**EXP-1 Result:** WER 46.47% ✗ (got WORSE from baseline 45.27%)  
**Issue:** Model over-predicts anusvara (ం) at word endings.  
**Solution:** Try stronger regularization via SpecAugment.

### Hypothesis
EXP-1's moderate augmentation allowed overfitting to the anusvara artifact.  
Stronger time masking (0.15 prob, 15-frame spans) + more attention dropout (0.15)  
will force the model to rely on broader context → should reduce the artifact.

### Hardware target
NVIDIA RTX A6000 · 49 GB VRAM · CUDA 12.4 · JupyterHub server

### Prerequisites (produced by Files 1 & 2)

| Artifact | Server path |
|---|---|
| Clean HuggingFace dataset | `/home/jovyan/work/telugu_asr_clean_dataset/` |
| Character vocabulary | `/home/jovyan/work/vocab.json` |

### Goal
Fine-tune XLS-R-300M with stronger SpecAugment. **Target: WER < 40%** (baseline: 45.27%).

### Key Differences from EXP-1
- `ATTENTION_DROPOUT`: 0.1 → **0.15** (more Transformer regularization)
- `MASK_TIME_PROB`: 0.1 → **0.15** (stronger time masking)
- `MASK_TIME_LENGTH`: 10 → **15** (longer mask spans)
- All else same: LR=1e-4, WARMUP=1000, EPOCHS=50, **FRESH TRAINING** (not resumed)

### Before running
Install PyTorch with CUDA 12.4 **in a terminal** (do this once, not inside the notebook):
```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
```
Then **Kernel → Restart** and run all cells top-to-bottom.


## Section 0 — Environment Setup

Installs all packages except torch (torch must be installed in terminal with the cu124 wheel).
Run once → **Kernel → Restart** → run all cells from top.

In [ ]:
# ── Section 0 — Environment Setup ─────────────────────────────────────────
# torch must be installed separately in terminal before running this notebook:
#   pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# After installing torch, restart the kernel, then run all cells.

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

# typing_extensions >= 4.10.0 required for TypeIs (used by transformers >= 4.46)
pip("--upgrade", "typing_extensions>=4.10.0")

pip(
    "--upgrade",
    "transformers>=4.46.0",   # Trainer API with processing_class, eval_strategy
    "accelerate>=1.1.0",      # Required by Trainer
    "datasets>=2.14,<3.0",    # <3.0: avoids torchcodec which needs FFmpeg
    "evaluate",               # WER / CER metrics
    "jiwer",                  # WER backend used by evaluate
    "soundfile",              # WAV decoding backend (replaces torchaudio)
    "librosa",                # Audio resampling; needed by datasets Audio feature
    "numpy",
)

print("Environment ready.")

## Section 1 — Imports & Logging

In [ ]:
# ── Section 1 — Imports & Logging ─────────────────────────────────────────
import logging
import os
import sys
import json
from dataclasses import dataclass
from typing import Dict, List, Union

import numpy as np
import torch
import evaluate
from datasets import load_from_disk, Audio
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2CTCTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# ── Dual-sink logging: stdout + file ──────────────────────────────────────
LOG_FILE = "training.log"
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8"),
    ],
)
logger = logging.getLogger("telugu_asr")

# Silence noisy third-party loggers
# numba floods DEBUG output when librosa triggers JIT compilation
# httpcore/httpx log every HuggingFace Hub request
for _noisy in ["numba", "httpcore", "httpx", "urllib3", "filelock",
               "datasets", "transformers", "huggingface_hub"]:
    logging.getLogger(_noisy).setLevel(logging.WARNING)

logger.info("Imports complete. torch=%s", torch.__version__)

## Section 2 — Hyperparameters (EXP-1)

Key changes from the baseline run (WER 45.27%):

| Parameter | Baseline | EXP-1 | Why |
|---|---|---|---|
| `LR` | 3e-4 | **1e-4** | Finer updates in late training; prevents overshooting |
| `WARMUP` | 500 | **1000** | ~1 full epoch warmup stabilises gradient norms before cosine decay |
| `MASK_FEAT_PROB` | 0.004 | **0.1** | **Biggest fix.** 0.004 ≈ off. Standard wav2vec2 uses 0.1 |
| `MASK_TIME_PROB` | 0.075 | **0.1** | Slightly stronger time masking |
| `EPOCHS` | 30 | **50** | More budget; early stopping with patience=8 prevents over-running |
| `early_stopping_patience` | 5 | **8** | Cosine tail can still yield improvements over 5–7 eval windows |

In [ ]:
# ── Section 2 — Hyperparameters (EXP-3: STRONGER SPECAUGMENT) ──────────────

# ── Paths (absolute — /home/jovyan/work is the persistent work directory) ─
MODEL_NAME          = "facebook/wav2vec2-xls-r-300m"
OUTPUT_DIR          = "/home/jovyan/work/wav2vec2-xls-r-300m-telugu-exp3"  # EXP-3
DATASET_PATH        = "/home/jovyan/work/telugu_asr_clean_dataset"
VOCAB_PATH          = "/home/jovyan/work/vocab.json"
PROCESSOR_SAVE_PATH = "/home/jovyan/work/telugu_wav2vec2_processor"

# ── Training schedule (EXP-3: Same as EXP-1 but stronger augmentation) ────
EPOCHS      = 50
BATCH_SIZE  = 8          # per-device; effective batch = 8 × 4 = 32
GRAD_ACCUM  = 4
LR          = 1e-4       # Keep EXP-1's LR
WARMUP      = 1000       # Keep EXP-1's warmup
EVAL_STEPS  = 500
SAVE_STEPS  = 500

# ── Misc ───────────────────────────────────────────────────────────────────
FP16        = torch.cuda.is_available()   # auto-detect
SPLIT_RATIO = 0.1                         # 10% eval
SEED        = 42

# ── Regularisation / SpecAugment (EXP-3: STRONGER than EXP-1) ─────────────
ATTENTION_DROPOUT = 0.15    # EXP-1: 0.1 → EXP-3: 0.15 (more dropout)
HIDDEN_DROPOUT    = 0.1
FEAT_PROJ_DROPOUT = 0.0
MASK_TIME_PROB    = 0.15    # EXP-1: 0.1 → EXP-3: 0.15 (stronger time masking)
MASK_TIME_LENGTH  = 15      # EXP-1: 10  → EXP-3: 15   (longer mask spans)
MASK_FEAT_PROB    = 0.1     # Keep EXP-1's value

NUM_INFERENCE_SAMPLES = 5

print("Hyperparameters ready (EXP-3: STRONGER SPECAUGMENT)."")
print(f"  Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  FP16                 : {FP16}")
print(f"  LR                   : {LR}  (same as EXP-1)")
print(f"  ATTENTION_DROPOUT    : {ATTENTION_DROPOUT}  (was 0.1 in EXP-1)")
print(f"  MASK_TIME_PROB       : {MASK_TIME_PROB}  (was 0.1 in EXP-1)")
print(f"  MASK_TIME_LENGTH     : {MASK_TIME_LENGTH}  (was 10 in EXP-1)")
print(f"  Output dir           : {OUTPUT_DIR}")


## Section 3 — GPU Diagnostics & TF32

In [ ]:
# ── Section 3 — GPU Diagnostics ────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    props  = torch.cuda.get_device_properties(0)

    # TF32 speeds up FP32 matmuls on Ampere+ GPUs (RTX A6000 is Ampere GA102)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True

    print("GPU DIAGNOSTICS")
    print(f"  PyTorch version  : {torch.__version__}")
    print(f"  CUDA version     : {torch.version.cuda}")
    print(f"  Device name      : {props.name}")
    print(f"  Total VRAM       : {props.total_memory / 1e9:.1f} GB")
    print(f"  Compute cap.     : {props.major}.{props.minor}")
    print(f"  FP16 enabled     : {FP16}")
    print("  TF32 enabled for matmul and cuDNN.")
else:
    DEVICE = torch.device("cpu")
    print("WARNING: No CUDA GPU detected — training will be very slow on CPU.")
    print("  If you expect a GPU, check:")
    print("    1. torch was installed with cu124 index URL")
    print("    2. nvidia-smi shows CUDA Driver >= 12.4")
    print("    3. Kernel was restarted after torch install")

print(f"\nDevice: {DEVICE}")

## Section 4 — Build Processor & Load Dataset

**Why build the processor from scratch?**  
HuggingFace's `Wav2Vec2Processor.from_pretrained()` validates the path as a Hub repo-id.  
Local paths with multiple slashes (e.g. `/home/jovyan/work/processor/`) trip that regex  
and raise `HFValidationError`. Building directly from components bypasses that entirely.

**Why `cast_column("audio", Audio(16000))`?**  
This tells the HuggingFace dataset to decode each WAV file and resample to 16 kHz  
on-the-fly when a sample is accessed. `prepare_dataset` then reads `batch["audio"]["array"]`  
— a clean float32 numpy waveform — with no manual file I/O needed.  
This is the same approach used in Files 1 and 2.

In [ ]:
# ── Section 4 — Build Processor & Load Dataset ────────────────────────────

# ── Verify prerequisites exist on this server ──────────────────────────────
assert os.path.isfile(VOCAB_PATH), (
    f"vocab.json not found at '{VOCAB_PATH}'.\n"
    "Run File 2 (02_model_prototyping_and_validation.ipynb) first."
)
assert os.path.isdir(DATASET_PATH), (
    f"Dataset not found at '{DATASET_PATH}'.\n"
    "Run File 1 (01_data_exploration_and_cleaning.ipynb) first."
)

# ── Feature extractor ─────────────────────────────────────────────────────
# Normalises raw 16 kHz waveforms to zero-mean / unit-variance input_values.
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16_000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

# ── CTC tokenizer from our custom vocab.json ──────────────────────────────
# Converts Telugu characters <-> integer IDs.
# word_delimiter_token='|' replaces spaces between words.
tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_PATH,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

# ── Bundle into Wav2Vec2Processor ─────────────────────────────────────────
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print("Processor built from scratch (no from_pretrained).")
print(f"  Vocab size   : {processor.tokenizer.vocab_size}")
print(f"  PAD token id : {processor.tokenizer.pad_token_id}")

# ── Load dataset from disk ────────────────────────────────────────────────
dataset = load_from_disk(DATASET_PATH)

# cast_column instructs HuggingFace to decode WAV files via soundfile
# and resample to 16 kHz automatically when each sample is accessed.
# This is the same approach used successfully in Files 1 & 2.
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

print(f"\nDataset loaded  : {len(dataset)} samples")
print(f"Columns         : {dataset.column_names}")

# ── Deterministic 90/10 train/eval split ──────────────────────────────────
split    = dataset.train_test_split(test_size=SPLIT_RATIO, seed=SEED)
train_ds = split["train"]
eval_ds  = split["test"]   # kept raw (with audio column) for inference in Section 12

print(f"Train split     : {len(train_ds)} samples")
print(f"Eval  split     : {len(eval_ds)} samples")

### Section 4b — Audio Sanity Check

Verify the dataset can decode one audio sample before the expensive full-dataset `map()`.  
If this cell fails, see the printed error message for guidance.

In [ ]:
# ── Section 4b — Audio Sanity Check ───────────────────────────────────────
print("Checking first training sample...")
try:
    _s   = train_ds[0]
    _aud = _s["audio"]
    print(f"  audio path    : {_aud.get('path', '<embedded bytes>')}")
    print(f"  array shape   : {_aud['array'].shape}")
    print(f"  sampling rate : {_aud['sampling_rate']} Hz")
    print(f"  transcription : {_s['transcription']}")
    print("  OK — audio is readable.")
except Exception as _e:
    print(f"  ERROR: {_e}")
    print()
    print("  Likely causes:")
    print("  1. Audio files have moved since the dataset was saved.")
    print("     Fix: re-run File 1 on this server to regenerate the dataset.")
    print("  2. soundfile / librosa not installed.")
    print("     Fix: re-run Section 0 (pip install cell) then restart kernel.")
    raise

## Section 5 — Prepare Dataset (Feature Extraction + Label Encoding)

In [ ]:
# ── Section 5 — Prepare Dataset ───────────────────────────────────────────

def prepare_dataset(batch: dict) -> dict:
    """
    Convert one raw dataset sample into model-ready tensors.

    Input columns : audio (dict: array, sampling_rate, path), transcription, audio_id
    Output columns: input_values (float32 array), input_length (int), labels (int list)

    Notes:
    - cast_column("audio", Audio(16000)) already decoded the WAV and resampled to 16 kHz.
      batch["audio"]["array"] is a float32 numpy waveform ready to use.
    - We do NOT pad here — the DataCollator pads dynamically per batch.
    - Spaces in transcription are replaced by '|' (word-boundary token) automatically
      by the CTC tokenizer.
    """
    audio = batch["audio"]

    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_values[0]    # shape: (num_frames,)

    batch["input_length"] = len(batch["input_values"])

    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch


COLS_TO_REMOVE = ["audio", "transcription", "audio_id"]

print("Preparing train split ...")
train_ds_prepared = train_ds.map(
    prepare_dataset,
    remove_columns=COLS_TO_REMOVE,
    num_proc=4,          # parallel workers; set to 1 if you see subprocess/import errors
    desc="Train features",
)

print("Preparing eval split ...")
eval_ds_prepared = eval_ds.map(
    prepare_dataset,
    remove_columns=COLS_TO_REMOVE,
    num_proc=4,
    desc="Eval features",
)

# Sort by input_length so consecutive batches contain similar-duration clips.
# This minimises padding waste and replaces the removed group_by_length=True
# argument (dropped in transformers >= 4.46).
train_ds_prepared = train_ds_prepared.sort("input_length")

print(f"\nTrain : {len(train_ds_prepared)} samples")
print(f"Eval  : {len(eval_ds_prepared)} samples")
print(f"Columns : {train_ds_prepared.column_names}")

## Section 6 — DataCollator: Dynamic Padding + −100 Label Masking

**CRITICAL:** Token-id `0` is the real Telugu character `ం`.  
We must use `-100` (not `0`) as the label pad so CTC loss skips padding positions.

In [ ]:
# ── Section 6 — DataCollator ───────────────────────────────────────────────

@dataclass
class DataCollatorCTCWithPadding:
    """
    Collate samples into a padded batch for CTC training.

    - input_values : right-padded to the longest waveform in the batch.
    - labels       : right-padded with -100 (CTC loss ignores -100 positions).

    IMPORTANT: token-id 0 is the real Telugu character ం.
    We must NOT use 0 as the label pad — use -100 so CTC loss skips it.
    """
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        # Pad input waveforms
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Pad label sequences; replace tokenizer-pad positions with -100
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# Quick sanity check: collate 2 samples
_test = data_collator([train_ds_prepared[0], train_ds_prepared[1]])
print("DataCollator ready.")
print(f"  input_values shape : {tuple(_test['input_values'].shape)}")
print(f"  labels shape       : {tuple(_test['labels'].shape)}")
print(f"  labels contain -100: {(_test['labels'] == -100).any().item()}")

## Section 7 — Load `facebook/wav2vec2-xls-r-300m`

XLS-R 300M was pre-trained on 436K hours of multilingual speech including Indian languages.  
We freeze the CNN feature encoder (7 convolutional blocks that process raw waveforms) because  
those weights are already excellent for audio feature extraction.  
The Transformer layers are the ones that will learn Telugu-specific patterns.

Gradient checkpointing recomputes activations on the backward pass: ~20% slower,  
~40% less VRAM — worth it even on the 49 GB A6000 to allow larger batches.

In [ ]:
# ── Section 7 — Load Model ─────────────────────────────────────────────────
VOCAB_SIZE = processor.tokenizer.vocab_size   # 67 for Telugu

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    # ── Regularisation ────────────────────────────────────────────────────
    attention_dropout=ATTENTION_DROPOUT,
    hidden_dropout=HIDDEN_DROPOUT,
    feat_proj_dropout=FEAT_PROJ_DROPOUT,
    # ── SpecAugment ───────────────────────────────────────────────────────
    mask_time_prob=MASK_TIME_PROB,
    mask_time_length=MASK_TIME_LENGTH,
    mask_feature_prob=MASK_FEAT_PROB,    # KEY FIX: was 0.004 (≈ off) in baseline
    # ── CTC head ──────────────────────────────────────────────────────────
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,              # replace inf CTC losses with 0 → prevents NaN
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=VOCAB_SIZE,
    ignore_mismatched_sizes=True,        # re-initialise lm_head to Telugu vocab size
)

# Freeze the CNN feature encoder — already well-trained on multilingual audio
model.freeze_feature_encoder()

# Gradient checkpointing: recompute activations on backward pass
model.gradient_checkpointing_enable()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model            : {MODEL_NAME}")
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}  ({100 * trainable / total:.1f}%)")
print(f"CNN encoder      : FROZEN")
print(f"Grad checkpointing: ENABLED")
print(f"Vocab size       : {VOCAB_SIZE}")

## Section 8 — WER & CER Metrics

In [ ]:
# ── Section 8 — Metrics ────────────────────────────────────────────────────
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids  = np.argmax(pred.predictions, axis=-1)
    label_ids = pred.label_ids

    # Replace -100 label-pad positions with pad_token_id before decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    # Log 3 sample predictions for qualitative inspection
    for i in range(min(3, len(pred_str))):
        logger.info("  REF : %s", label_str[i])
        logger.info("  PRED: %s", pred_str[i])
    logger.info("WER: %.4f  |  CER: %.4f", wer, cer)

    return {"wer": wer, "cer": cer}


print("WER metric :", wer_metric)
print("CER metric :", cer_metric)

## Section 9 — TrainingArguments

In [ ]:
# ── Section 9 — TrainingArguments ─────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── Schedule ──────────────────────────────────────────────────────────
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,    # effective batch = 32
    learning_rate=LR,
    warmup_steps=WARMUP,
    lr_scheduler_type="cosine",                # cosine decay after warmup

    # ── Precision & speed ─────────────────────────────────────────────────
    fp16=FP16,
    dataloader_num_workers=4,
    # group_by_length removed in transformers >= 4.46; we pre-sort by input_length

    # ── Eval & saving ─────────────────────────────────────────────────────
    eval_strategy="steps",         # renamed from evaluation_strategy in >= 4.46
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,       # lower WER = better

    # ── Logging ───────────────────────────────────────────────────────────
    logging_steps=100,
    report_to="none",

    # ── Misc ──────────────────────────────────────────────────────────────
    seed=SEED,
    remove_unused_columns=False,   # keep input_length column
    label_names=["labels"],
    push_to_hub=False,
)

print("TrainingArguments configured.")
print(f"  Epochs            : {EPOCHS}")
print(f"  Effective batch   : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR                : {LR}")
print(f"  Warmup steps      : {WARMUP}")
print(f"  LR scheduler      : cosine")
print(f"  FP16              : {FP16}")
print(f"  Best model metric : wer (lower is better)")

## Section 10 — Initialize Trainer & Run Training (EXP-1)

In [ ]:
# ── Section 10 — Trainer + Train ──────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_prepared,
    eval_dataset=eval_ds_prepared,
    processing_class=processor.feature_extractor,  # renamed from tokenizer in >= 4.46
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=8,      # EXP-1: was 5 (too aggressive with cosine tail)
            early_stopping_threshold=0.005, # EXP-1: was 0.001 (~18 words, meaningful)
        )
    ],
)

logger.info("=" * 60)
logger.info("STARTING EXP-1 TRAINING")
logger.info("  Model    : %s", MODEL_NAME)
logger.info("  LR       : %g  |  Epochs : %d  |  Eff. batch : %d",
            LR, EPOCHS, BATCH_SIZE * GRAD_ACCUM)
logger.info("  MASK_FEAT_PROB : %.3f  (baseline was 0.004 — key fix)", MASK_FEAT_PROB)
logger.info("=" * 60)

try:
    trainer.train()
except KeyboardInterrupt:
    logger.warning("Training interrupted — saving current checkpoint.")
    trainer.save_model(os.path.join(OUTPUT_DIR, "checkpoint-interrupted"))
    raise
except Exception as exc:
    logger.error("Training failed: %s", exc, exc_info=True)
    raise

## Section 11 — Save Final Model & Run Evaluation

In [ ]:
# ── Section 11 — Save + Evaluate ──────────────────────────────────────────
logger.info("Saving EXP-1 model + processor to: %s", OUTPUT_DIR)

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(PROCESSOR_SAVE_PATH)   # also update the shared processor dir

# Remove NotebookProgressCallback before standalone evaluate().
# It requires on_train_begin() which is not called outside of train().
try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

logger.info("Running final evaluation on eval set...")
final_metrics = trainer.evaluate()

final_wer = final_metrics.get("eval_wer", float("nan"))
final_cer = final_metrics.get("eval_cer", float("nan"))

print("=" * 60)
print("EXP-1 FINAL RESULTS")
print(f"  WER : {final_wer:.4f}  ({'✓' if final_wer < 0.40 else '✗'} target < 0.40)")
print(f"  CER : {final_cer:.4f}")
print("=" * 60)

metrics_path = os.path.join(OUTPUT_DIR, "exp1_eval_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(final_metrics, f, indent=2)
print(f"Metrics saved to: {metrics_path}")

## Section 12 — Inference on 5 Eval Samples

In [ ]:
# ── Section 12 — Inference ────────────────────────────────────────────────
inference_model  = trainer.model.eval()
inference_device = next(inference_model.parameters()).device

print("=" * 60)
print("EXP-1 INFERENCE — 5 SAMPLE PREDICTIONS")
print("=" * 60)

for i in range(min(NUM_INFERENCE_SAMPLES, len(eval_ds))):
    sample = eval_ds[i]
    inputs = processor(
        sample["audio"]["array"],
        sampling_rate=16_000,
        return_tensors="pt",
        padding=True,
    )
    input_values   = inputs.input_values.to(inference_device)
    attention_mask = inputs.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(inference_device)

    with torch.no_grad():
        kwargs = {"input_values": input_values}
        if attention_mask is not None:
            kwargs["attention_mask"] = attention_mask
        logits = inference_model(**kwargs).logits

    predicted = processor.batch_decode(torch.argmax(logits, dim=-1))[0]
    print(f"\n[{i+1}] REF : {sample['transcription']}")
    print(f"     PRED: {predicted}")

print("\n" + "=" * 60)

---
## Section 13 — EXP-2: Resume from EXP-1 + Unfreeze CNN

**Run only after EXP-1 completes.**

**Rationale:**  
XLS-R's CNN encoder was pre-trained on 128 languages but never specifically on  
Dravidian vowel harmony or Telugu retroflex stops. A very-low-LR pass (1e-5)  
with the CNN unfrozen lets it adapt to Telugu acoustics without destroying  
the Transformer layers fine-tuned in EXP-1.  
A constant scheduler keeps updates steady across the full pass.

**Expected WER:** ~35–39 %  
**Risk:** If LR is still too high the CNN can collapse. Mitigated by the 1e-5 setting.

In [ ]:
# ── Section 13 — EXP-2 ────────────────────────────────────────────────────
import glob

EXP1_DIR = "/home/jovyan/work/wav2vec2-xls-r-300m-telugu-exp1"
EXP2_DIR = "/home/jovyan/work/wav2vec2-xls-r-300m-telugu-exp2"
os.makedirs(EXP2_DIR, exist_ok=True)

# ── Find the best checkpoint from EXP-1 ───────────────────────────────────
checkpoints = sorted(glob.glob(f"{EXP1_DIR}/checkpoint-*"), key=os.path.getmtime)
assert checkpoints, (
    f"No checkpoints found in '{EXP1_DIR}'.\n"
    "Run EXP-1 (Sections 0-12) first."
)
RESUME_FROM = checkpoints[-1]    # latest = best (load_best_model_at_end=True saved it)
print(f"Resuming from: {RESUME_FROM}")

# ── Load model from EXP-1 checkpoint ─────────────────────────────────────
exp2_model = Wav2Vec2ForCTC.from_pretrained(RESUME_FROM)

# Freeze everything first, then selectively unfreeze the CNN encoder
exp2_model.freeze_feature_encoder()
for param in exp2_model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = True

_frozen   = sum(p.numel() for p in exp2_model.parameters() if not p.requires_grad)
_total_e2 = sum(p.numel() for p in exp2_model.parameters())
print(f"Trainable params: {_total_e2 - _frozen:,}  /  {_total_e2:,}")
print("CNN encoder : UNFROZEN for Telugu acoustic adaptation")

# ── EXP-2 TrainingArguments ───────────────────────────────────────────────
exp2_args = TrainingArguments(
    output_dir=EXP2_DIR,
    num_train_epochs=20,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=1e-5,            # 10x lower than EXP-1 — gentle CNN adaptation
    warmup_steps=0,
    lr_scheduler_type="constant",  # no decay — keeps updates steady
    fp16=FP16,
    dataloader_num_workers=4,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=100,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
    label_names=["labels"],
    push_to_hub=False,
)

exp2_trainer = Trainer(
    model=exp2_model,
    args=exp2_args,
    train_dataset=train_ds_prepared,
    eval_dataset=eval_ds_prepared,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=6,
            early_stopping_threshold=0.005,
        )
    ],
)

logger.info("=" * 60)
logger.info("STARTING EXP-2 (unfrozen CNN, LR=1e-5, constant schedule)")
logger.info("=" * 60)

exp2_trainer.train()

# ── Save + evaluate ───────────────────────────────────────────────────────
exp2_trainer.save_model(EXP2_DIR)
processor.save_pretrained(EXP2_DIR)

try:
    from transformers.utils.notebook import NotebookProgressCallback
    exp2_trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

exp2_metrics = exp2_trainer.evaluate()
print("\nEXP-2 RESULTS")
print(f"  WER : {exp2_metrics['eval_wer']:.4f}  ({'✓' if exp2_metrics['eval_wer'] < 0.40 else '✗'})")
print(f"  CER : {exp2_metrics['eval_cer']:.4f}")

with open(os.path.join(EXP2_DIR, "exp2_eval_metrics.json"), "w") as f:
    json.dump(exp2_metrics, f, indent=2)
print(f"EXP-2 model saved to {EXP2_DIR}")

---
## Section 14 — EXP-3: Stronger SpecAugment (Fallback — only if EXP-1/2 WER > 40%)

**Run only if EXP-1 AND EXP-2 both finished with WER > 40%.**

**Rationale:**  
Increasing `mask_time_prob` (0.1→0.15) and `mask_time_length` (10→15) forces  
the model to rely on broader context — improves generalisation in low-resource settings.  
Extra attention dropout (0.15) adds Transformer-level regularisation.  
Must be trained **from scratch** — stronger augmentation must apply from step 0.

**Expected WER:** ~37–41%

**How to run:**  
1. Copy the values below into Section 2 (cell-s2-hparams).  
2. Change `OUTPUT_DIR` to `/home/jovyan/work/wav2vec2-xls-r-300m-telugu-exp3`.  
3. **Kernel → Restart & Run All** — do NOT resume from EXP-1/2.

In [ ]:
# ── Section 14 — EXP-3 Parameter Overrides ────────────────────────────────
# This cell only PRINTS the overrides. To actually run EXP-3:
#   1. Paste these values into Section 2 (Hyperparameters cell).
#   2. Change OUTPUT_DIR to the exp3 path below.
#   3. Kernel -> Restart & Run All (fresh training from scratch).

EXP3_OVERRIDES = {
    "OUTPUT_DIR"        : "/home/jovyan/work/wav2vec2-xls-r-300m-telugu-exp3",
    "EPOCHS"            : 50,
    "LR"                : 1e-4,
    "WARMUP"            : 1000,
    "ATTENTION_DROPOUT" : 0.15,   # 0.1 → 0.15 (more Transformer dropout)
    "HIDDEN_DROPOUT"    : 0.1,
    "FEAT_PROJ_DROPOUT" : 0.0,
    "MASK_TIME_PROB"    : 0.15,   # 0.1 → 0.15 (stronger time masking)
    "MASK_TIME_LENGTH"  : 15,     # 10  → 15   (longer mask spans)
    "MASK_FEAT_PROB"    : 0.1,
}

print("EXP-3 parameter overrides (apply to Section 2, then Restart & Run All):")
for k, v in EXP3_OVERRIDES.items():
    print(f"  {k:25s} = {v}")

print("\nIMPORTANT: Do NOT resume from EXP-1/2 checkpoints.")
print("Fresh training ensures stronger augmentation applies from step 0.")